# Three-path interferometer – live fringe simulator

Interactive model of the camera pattern from the three-path PBS interferometer (λ₀ = 810 nm).
The three beams reach the camera **side by side** along x (A left, B middle, C right) and slightly tilted toward each other, so where they overlap they form straight fringes.

| path | launching optic (stage + tilt) | phase shifter |
|---|---|---|
| A | PBS1 (reflected arm) | slide θ_A |
| B | PBS3 (reflected arm) | slide θ_B |
| C | M2 (right-hand mirror) | – (reference arm) |

**What each control does**
- **Slide θ** – tilting a microscope slide changes that path's phase (and shifts the beam slightly sideways).
- **Stage** – translating PBS1 / PBS3 / M2 along the incoming beam moves that beam sideways on the camera *and* lengthens the path by the same amount. At 810 nm, 1 µm of stage is ≈ 7.8 rad, so stages are for overlap; set phases with the slides. Untick *stage moves change path length* to see only the position effect.
- **Tilt x / y** – tilting the optic by δ steers the beam by 2δ: this sets the fringe spacing and orientation, and moves the beam by 2δ·L on the camera (L = lever arm to the camera, set in `Setup.lever_mm`).
- **Drift step / Record reference** – record the fringe profile, kick the phases, then turn the slides to bring the fringes back onto the dashed reference.

The physics lives in `interferometer_sim.py` (`Setup`), so the same model can be used in analysis scripts. Assumptions: scalar Gaussian beams of equal amplitude, analyzer projecting all paths onto one polarization, BK7-like slide index.


In [ ]:
import io
import numpy as np
import ipywidgets as W
from IPython.display import display
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

from interferometer_sim import Setup, PATHS, LAUNCH_OPTIC, slide_slope, slide_walk

setup = Setup()
state = {"ref": None}
FOV, NPIX = 3.0, 241
COLORS = {"A": "#4fc3f7", "B": "#ffd54f", "C": "#ef5350"}

# ---------------------------------------------------------------- widgets
def fslider(desc, v, lo, hi, step, fmt):
    return W.FloatSlider(value=v, min=lo, max=hi, step=step, description=desc,
                         readout_format=fmt, continuous_update=True,
                         style={"description_width": "110px"}, layout=W.Layout(width="420px"))

thA = fslider("slide θ_A (°)", setup.theta_A_deg, 0, 30, 0.001, ".3f")
thB = fslider("slide θ_B (°)", setup.theta_B_deg, 0, 30, 0.001, ".3f")

stage, tx, ty, blk = {}, {}, {}, {}
tabs = []
for p in PATHS:
    L = setup.launch[p]
    stage[p] = fslider(f"{LAUNCH_OPTIC[p]} stage (mm)", L.stage_mm, -1.5, 1.5, 0.0005, ".4f")
    tx[p] = fslider(f"{LAUNCH_OPTIC[p]} tilt x (mrad)", L.tilt_x_mrad, -2, 2, 0.005, ".3f")
    ty[p] = fslider(f"{LAUNCH_OPTIC[p]} tilt y (mrad)", L.tilt_y_mrad, -2, 2, 0.005, ".3f")
    blk[p] = W.Checkbox(value=L.blocked, description=f"block path {p}")
    tabs.append(W.VBox([stage[p], tx[p], ty[p], blk[p]]))
beam_tabs = W.Tab(children=tabs)
for i, p in enumerate(PATHS):
    beam_tabs.set_title(i, f"{p}  ({LAUNCH_OPTIC[p]})")

lam = W.BoundedFloatText(value=setup.lam_nm, min=300, max=1200, step=1, description="λ₀ (nm)")
thick = W.BoundedFloatText(value=setup.slide_t_mm, min=0.05, max=5, step=0.05, description="slide t (mm)")
waist = W.BoundedFloatText(value=setup.waist_mm, min=0.1, max=5, step=0.05, description="waist (mm)")
stage_ph = W.Checkbox(value=setup.stage_changes_phase, description="stage moves change path length")

b_drift = W.Button(description="Drift step", tooltip="Add a random phase kick (σ = 0.5 rad) to every path")
b_undrift = W.Button(description="Clear drift")
b_ref = W.Button(description="Record reference", button_style="info")
b_clear = W.Button(description="Clear reference")
b_reset = W.Button(description="Reset all", button_style="warning")

info = W.HTML()
img = W.Image(format="png")

# ---------------------------------------------------------------- figure
fig = Figure(figsize=(11, 4.4), dpi=90)
FigureCanvasAgg(fig)
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.35], wspace=0.25)
ax_cam = fig.add_subplot(gs[0, 0])
ax_pro = fig.add_subplot(gs[0, 1])
x, y, I0 = setup.camera(FOV, NPIX)
X, Y = np.meshgrid(x, y)
im = ax_cam.imshow(I0, extent=[-FOV, FOV, -FOV, FOV], origin="lower", cmap="inferno", vmin=0, vmax=1)
ax_cam.set_title("camera (simulated)")
ax_cam.set_xlabel("x (mm)"); ax_cam.set_ylabel("y (mm)")
ax_cam.axhline(0, color="w", lw=0.6, ls=":", alpha=0.6)
markers = {p: ax_cam.plot([], [], "+", color=COLORS[p], ms=10, mew=2)[0] for p in PATHS}
labels = {p: ax_cam.text(0, 0, p, color=COLORS[p], fontsize=10, fontweight="bold") for p in PATHS}
line_now, = ax_pro.plot(x, I0[NPIX // 2], color="#5b45e0", lw=2, label="now")
line_ref, = ax_pro.plot([], [], color="#0d8774", lw=1.5, ls="--", label="reference")
env_lines = {p: ax_pro.plot(x, 0 * x, color=COLORS[p], lw=1, alpha=0.7, label=f"|E_{p}|²")[0] for p in PATHS}
ax_pro.set_xlim(-FOV, FOV); ax_pro.set_ylim(0, 1.05)
ax_pro.set_xlabel("x (mm)"); ax_pro.set_ylabel("I / I_max")
ax_pro.set_title("cross-section at y = 0")
ax_pro.grid(alpha=0.3); ax_pro.legend(loc="upper right", fontsize=8)


def read_widgets():
    setup.theta_A_deg, setup.theta_B_deg = thA.value, thB.value
    setup.lam_nm, setup.slide_t_mm, setup.waist_mm = lam.value, thick.value, waist.value
    setup.stage_changes_phase = stage_ph.value
    for p in PATHS:
        L = setup.launch[p]
        L.stage_mm, L.tilt_x_mrad, L.tilt_y_mrad, L.blocked = stage[p].value, tx[p].value, ty[p].value, blk[p].value


def redraw(*_):
    read_widgets()
    Imax = setup.max_intensity(X, Y)
    I = setup.intensity(X, Y) / Imax
    im.set_data(I)
    prof = I[NPIX // 2]
    line_now.set_ydata(prof)
    for p in PATHS:
        b = setup.beam(p)
        on = not b["blocked"]
        markers[p].set_data([b["x0"]], [b["y0"]]); markers[p].set_visible(on)
        labels[p].set_position((b["x0"] + 0.12, b["y0"] + 0.15)); labels[p].set_visible(on)
        env = b["amp"]**2 * np.exp(-2 * ((x - b["x0"])**2 + b["y0"]**2) / setup.waist_mm**2) / Imax
        env_lines[p].set_ydata(env); env_lines[p].set_visible(on)
    if state["ref"] is not None:
        line_ref.set_data(x, state["ref"])
        rms = np.sqrt(np.mean((prof - state["ref"])**2))
        ref_txt = f"RMS difference from reference: <b>{rms:.4f}</b>"
    else:
        line_ref.set_data([], [])
        ref_txt = "no reference recorded"
    buf = io.BytesIO(); fig.savefig(buf, format="png"); img.value = buf.getvalue()

    ph = setup.phases()
    fp = setup.fringe_periods()
    fp_txt = ", ".join(f"{k}: {v:.3f} mm" for k, v in fp.items()) or "–"
    ang = ", ".join(f"{p}: {setup.beam(p)['angle_x_mrad']:+.2f}" for p in PATHS)
    info.value = (
        "<div style='font-family:monospace;font-size:13px;line-height:1.6'>"
        f"φ_A = {ph['A']:.3f} rad &nbsp; φ_B = {ph['B']:.3f} rad &nbsp; φ_C = {ph['C']:.3f} rad (mod 2π)<br>"
        f"slide dφ/dθ: A {slide_slope(setup.theta_A_deg, setup.lam_nm, setup.slide_t_mm):.1f}, "
        f"B {slide_slope(setup.theta_B_deg, setup.lam_nm, setup.slide_t_mm):.1f} rad/° &nbsp; "
        f"slide beam walk: A {1e3*slide_walk(setup.theta_A_deg, setup.lam_nm, setup.slide_t_mm):.0f} µm, "
        f"B {1e3*slide_walk(setup.theta_B_deg, setup.lam_nm, setup.slide_t_mm):.0f} µm<br>"
        f"beam angles x (mrad): {ang}<br>"
        f"fringe periods: {fp_txt}<br>"
        f"stage phase: 1 µm of stage = {2*np.pi*1e3/setup.lam_nm:.2f} rad<br>{ref_txt}</div>"
    )


def on_drift(_):
    rng = np.random.default_rng()
    for p in PATHS:
        setup.launch[p].drift_rad += rng.normal(0, 0.5)
    redraw()


def on_undrift(_):
    for p in PATHS:
        setup.launch[p].drift_rad = 0.0
    redraw()


def on_ref(_):
    state["ref"] = line_now.get_ydata().copy(); redraw()


def on_clear(_):
    state["ref"] = None; redraw()


def on_reset(_):
    d = Setup()
    thA.value, thB.value = d.theta_A_deg, d.theta_B_deg
    lam.value, thick.value, waist.value, stage_ph.value = d.lam_nm, d.slide_t_mm, d.waist_mm, d.stage_changes_phase
    for p in PATHS:
        stage[p].value, tx[p].value, ty[p].value, blk[p].value = 0.0, d.launch[p].tilt_x_mrad, d.launch[p].tilt_y_mrad, False
        setup.launch[p].drift_rad = 0.0
    state["ref"] = None
    redraw()


for w in [thA, thB, lam, thick, waist, stage_ph] + [d[p] for d in (stage, tx, ty, blk) for p in PATHS]:
    w.observe(redraw, names="value")
b_drift.on_click(on_drift); b_undrift.on_click(on_undrift)
b_ref.on_click(on_ref); b_clear.on_click(on_clear); b_reset.on_click(on_reset)

controls = W.VBox([
    W.HTML("<b>Phase shifters (microscope slides)</b>"), thA, thB,
    W.HTML("<b>Launch optics</b> – stage translates the optic, tilt steers its beam"), beam_tabs,
    W.HTML("<b>Setup</b>"), W.HBox([lam, thick, waist]), stage_ph,
    W.HBox([b_drift, b_undrift, b_ref, b_clear, b_reset]),
])
redraw()
display(W.VBox([img, info, controls]))
